# Phase 3, experiment 7: model diversity and a simple ensemble

**Why this experiment:** Three changes to the feature set (degree, neighbour means, dropping drifting
features) left the score where the default LightGBM on raw features put it. The single model is also
unstable from one time period to the next. Different kinds of model make different mistakes, so
averaging them usually gives a ranking that is more stable than any one of them.

**What is tested:** Three model types on the same 165 raw features and the same three folds:

| Name | Model | Settings |
|---|---|---|
| lgbm | LightGBM (gradient boosting) | library defaults: 100 trees, learning rate 0.1, 31 leaves |
| xgb | XGBoost (gradient boosting) | 300 trees, learning rate 0.1, maximum depth 6 |
| rf | Random Forest | 500 trees |

LightGBM and XGBoost both build trees one after another, each correcting the errors of the previous
ones, but they grow the trees differently. A Random Forest builds its trees independently on random
samples of rows and features and averages them, so it behaves quite differently from the other two.

**How the models are combined: rank averaging.** Each model's predictions are turned into ranks
(lowest score = rank 1, and so on), and the ranks are averaged. The competition metric, ROC-AUC,
depends only on the order of the predictions, so ranks keep everything that matters. Ranking also
puts the three models on the same scale, so a model that gives very extreme probabilities does not
dominate the average.

**Decisions:** keep all 165 raw features (dropping the six drifting ones lowered both
the fold mean and the public leaderboard); try model diversity next, with settings near the library
defaults. Tuning is left for a later experiment.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

raw_cols = [c for c in train.columns if c.startswith("feat_")]

# Labeled rows only (1 = illicit, 0 = licit)
labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

## 2. Folds and models

In [ ]:
# Same three expanding-window folds as notebook 07
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]

def split_fold(df, fold):
    # Return the (training rows, validation rows) of df for one fold.
    tr = df[df["time_step"] <= fold["train_last"]]
    va = df[(df["time_step"] > fold["train_last"]) & (df["time_step"] <= fold["valid_last"])]
    return tr, va

def make_model(name):
    # Return a new, untrained model of the given type. A new one is needed for every fit.
    if name == "lgbm":
        return lgb.LGBMClassifier(random_state=SEED, verbose=-1)
    if name == "xgb":
        return xgb.XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                 random_state=SEED, n_jobs=-1)
    if name == "rf":
        # n_jobs=-1 uses all CPU cores, which makes the forest much faster to train
        return RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)
    raise ValueError("unknown model name: " + name)

MODEL_NAMES = ["lgbm", "xgb", "rf"]

def rank_average(pred_table, names):
    # Average the ranks of the chosen models' predictions.
    # pred_table has one column of predictions per model.
    # rank() turns scores into positions (1 = lowest score)
    # divide by the number of rows puts every model on the same 0-to-1 scale before averaging.
    ranks = pred_table[names].rank() / len(pred_table)
    return ranks.mean(axis=1)

## 3. Score every model and every combination on the folds

For each fold, the three models are trained on the fold's training steps and predict its validation
steps. The combinations are then scored from those same predictions.

In [3]:
# The single models and the combinations to score
COMBOS = {
    "lgbm": ["lgbm"],
    "xgb": ["xgb"],
    "rf": ["rf"],
    "lgbm+xgb": ["lgbm", "xgb"],
    "lgbm+rf": ["lgbm", "rf"],
    "xgb+rf": ["xgb", "rf"],
    "lgbm+xgb+rf": ["lgbm", "xgb", "rf"],
}

score_rows = []
fold_preds = {}   # fold name -> table of validation predictions (one column per model)

for fold in FOLDS:
    tr, va = split_fold(labeled, fold)

    preds = pd.DataFrame(index=va.index)
    for name in MODEL_NAMES:
        model = make_model(name)
        model.fit(tr[raw_cols], tr["label"])
        preds[name] = model.predict_proba(va[raw_cols])[:, 1]
        print("done:", fold["name"], "/", name)
    fold_preds[fold["name"]] = preds

    for combo_name, names in COMBOS.items():
        combined = rank_average(preds, names)
        score_rows.append({"fold": fold["name"], "model": combo_name,
                           "val_auc": roc_auc_score(va["label"], combined)})

scores = pd.DataFrame(score_rows)

done: fold 1 / lgbm
done: fold 1 / xgb
done: fold 1 / rf
done: fold 2 / lgbm
done: fold 2 / xgb
done: fold 2 / rf
done: fold 3 / lgbm
done: fold 3 / xgb
done: fold 3 / rf


## 4. Results

### 4.1 AUC per fold and the average

In [4]:
fold_names = [f["name"] for f in FOLDS]
fold_table = scores.pivot(index="model", columns="fold", values="val_auc").loc[list(COMBOS.keys())]
fold_table["mean"] = fold_table[fold_names].mean(axis=1)
fold_table["min"] = fold_table[fold_names].min(axis=1)
fold_table.round(4)

fold,fold 1,fold 2,fold 3,mean,min
model,,,,,
lgbm,0.9888,0.9572,0.9965,0.9808,0.9572
xgb,0.9920,0.9267,0.9966,0.9717,0.9267
rf,0.9898,0.8715,0.9950,0.9521,0.8715
lgbm+xgb,0.9927,0.9454,0.9969,0.9783,0.9454
lgbm+rf,0.9923,0.9219,0.9969,0.9704,0.9219
xgb+rf,0.9919,0.8967,0.9970,0.9619,0.8967
lgbm+xgb+rf,0.9930,0.9268,0.9973,0.9723,0.9268


### 4.2 How similar are the models' predictions?

Averaging helps most when the models disagree. This is the rank correlation between the models'
validation predictions (1.0 = identical ordering), averaged over the three folds.

In [5]:
corr_sum = None
for name in fold_names:
    corr = fold_preds[name].corr(method="spearman")   # Spearman = correlation of the ranks
    corr_sum = corr if corr_sum is None else corr_sum + corr
(corr_sum / len(fold_names)).round(3)

,lgbm,xgb,rf
lgbm,1.000,0.873,0.81
xgb,0.873,1.000,0.79
rf,0.810,0.790,1.00


### 4.3 AUC on step 24, the hardest step (from fold 2)

In [6]:
tr2, va2 = split_fold(labeled, FOLDS[1])
in_step_24 = (va2["time_step"] == 24).values

step24 = {}
for combo_name, names in COMBOS.items():
    combined = rank_average(fold_preds["fold 2"], names)
    step24[combo_name] = roc_auc_score(va2["label"][in_step_24], combined[in_step_24])
pd.Series(step24, name="auc_step_24").round(4)

lgbm           0.7866
xgb            0.6509
rf             0.4633
lgbm+xgb       0.7413
lgbm+rf        0.6395
xgb+rf         0.5400
lgbm+xgb+rf    0.6445
Name: auc_step_24, dtype: float64

## 5. Retrain on all labeled rows and save a submission

All three models are retrained on steps 1 to 35 and their test predictions are rank averaged.

In [ ]:
test_preds = pd.DataFrame(index=test.index)
for name in MODEL_NAMES:
    model = make_model(name)
    model.fit(labeled[raw_cols], labeled["label"])
    test_preds[name] = model.predict_proba(test[raw_cols])[:, 1]
    print("done:", name)

ENSEMBLE_MODELS = ["lgbm", "xgb", "rf"]   
test_pred = rank_average(test_preds, ENSEMBLE_MODELS)

submission = pd.DataFrame({"index": test["index"], "target": test_pred.values})

assert len(submission) == 15329, "wrong number of rows"
assert list(submission.columns) == ["index", "target"], "wrong columns"
assert submission["target"].notna().all(), "NaN in predictions"
assert (submission["index"].values == sample_sub["index"].values).all(), "index does not match sample submission"

SUB_PATH = "../submissions/sub05_ensemble.csv"
submission.to_csv(SUB_PATH, index=False)
print("Saved", SUB_PATH)

done: lgbm
done: xgb
done: rf
Saved ../submissions/sub05_ensemble.csv


## Results

- **How do the three single models compare?** On folds 1 and 3 they are close: 0.9888 to 0.9920 on
  fold 1 and 0.9950 to 0.9966 on fold 3, with XGBoost slightly ahead on both. On fold 2 they differ a
  lot: LightGBM 0.9572, XGBoost 0.9267, Random Forest 0.8715. LightGBM has the best mean (0.9808),
  then XGBoost (0.9717), then Random Forest (0.9521).
- **Does rank averaging beat the best single model?** It depends on the fold. On folds 1 and 3 the
  three-model average is the best of all seven options (0.9930 and 0.9973), ahead of LightGBM alone by
  0.0042 and 0.0008. On fold 2 it scores 0.9268, which is 0.030 below LightGBM alone. Its mean
  (0.9723) is therefore lower than LightGBM's (0.9808). LightGBM + XGBoost is in between (mean 0.9783).
- **How similar are the models' predictions?** The rank correlations are 0.87 (LightGBM and XGBoost),
  0.81 (LightGBM and Random Forest) and 0.79 (XGBoost and Random Forest), so the models do order the
  rows differently and there is diversity to average over.
- **What happens on step 24?** This step explains the fold 2 result. LightGBM scores 0.787, XGBoost
  0.651 and Random Forest 0.463. A score below 0.5 means Random Forest ranks the illicit rows in that
  step as less suspicious than the licit ones. Averaging LightGBM with the other two pulls its score
  on this step down to 0.645.
- **Summary:** in periods that resemble the training data, the ensemble is slightly better than any
  single model. In the one period where a new kind of illicit transaction appears, LightGBM alone copes
  much better than the other two models. This rests on a single step, and earlier notebooks showed that
  LightGBM's score on step 24 is itself unstable.
- **Decision: which models to use going forward?** LightGBM alone. The ensemble lowered both the fold
  mean (by 0.0085) and the public leaderboard (by 0.0058).
- **Public leaderboard AUC for `sub05_ensemble.csv`:** 0.94081, against 0.94660 for LightGBM alone
  (`sub01_baseline.csv`). That is 0.0058 lower, the largest drop of any submission so far. The direction
  matches the fold mean (0.9723 for the ensemble against 0.9808 for LightGBM), and suggests the test
  period contains new kinds of illicit behaviour that XGBoost and Random Forest handle worse than
  LightGBM, as on step 24.
